In [ ]:
import sys
import base64
from ollama import chat
from pathlib import Path
sys.path.append('..')
from utils.alarm import alarm
# Pass in the path to the image
# path = input('Please enter the path to the image: ')
def log_stream(response):
    print("Starting stream...\n")
    in_thinking = False
    content = ''
    thinking = ''

    for chunk in response:
        # print(chunk)
        if chunk.message.thinking:
            if not in_thinking:
                in_thinking = True
                print('Thinking:\n', end='', flush=True)
            print(chunk.message.thinking, end='', flush=True)
            # accumulate the partial thinking 
            thinking += chunk.message.thinking
        elif chunk.message.content != '':
            # if in_thinking:
                # in_thinking = False
            # alarm()
            if content == '':
                print('\n\nAnswer:\n', end='', flush=True)
            print(chunk.message.content, end='', flush=True)
            # accumulate the partial content
            content += chunk.message.content

# You can also pass in base64 encoded image data
# img = base64.b64encode(Path(path).read_bytes()).decode()
# or the raw bytes
# img = Path(path).read_bytes()
path = './template_images/template_1.jpeg'
# image = Path('./template_images/template_1.jpeg').read_bytes()
# img = base64.b64encode(Path(path).read_bytes()).decode()
f = open(path, 'rb')
img = f.read()
f.close()
response = chat(
  model='qwen3-vl:4b',
  messages=[
    {
      'role': 'user',
      'content': 'I need you to convert the image into a pure html and css code. Do not include any text explanation, only return the code. Here is the image:',
      'images': [img],
    }
  ],
  stream=True,
  think=False
)

log_stream(response)


Starting stream...

Thinking:
We are going to create a resume in the style of the given image.
 The design is inspired by the Pirates of the Caribbean character Jack Sparrow, but we'll make it a clean, modern resume.

 Steps:
 1. We'll structure the HTML with appropriate sections: header, main content (with columns for different parts), and footer.
 2. The image is split into two main columns: left for personal details and right for the rest.
 3. We'll use a CSS reset and a modern, clean font (like Arial or Helvetica) but we can use Google Fonts if needed.
 4. The design uses a dark gray header and light gray background for the main content, with blue for some accents.

 However, note: The image has some specific design elements:
   - The name "Jack Sparrow" at the top with "Captain" below.
   - Left sidebar: 
        - Profile picture (circular)
        - About me
        - Personal info (nationality, etc.)
        - Areas of specialization
        - Interests (with icons? but we'll u

In [16]:
from reportlab.lib.pagesizes import letter
from typing import Union, Literal
from dataclasses import dataclass
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, HRFlowable, Flowable
from reportlab.lib import colors

doc = SimpleDocTemplate("test.pdf", pagesize=letter, topMargin=0, leftMargin=0, rightMargin=0, bottomMargin=0)
total_width = letter[0]  # Adjusting for margins
resume = []

@dataclass
class Direction:
    left: int
    right: int
    top: int
    bottom: int

class Row:
    def __init__(self,items: Union[list, str], full_width: float = letter[0], orientation: Literal['ROW', 'COLUMN'] = 'ROW', padding: Direction = Direction(0,0,0,0), margin: Direction = Direction(0,0,0,0), align: Literal['LEFT', 'CENTER', 'RIGHT', 'BETWEEN', 'AROUND']='LEFT'):
        self.orientation = orientation
        self.full_width = full_width
        self.padding = padding
        self.margin = margin
        self.align = align
        self.items = items
        pass
    
    def render(self):
        items = []
        if isinstance(self.items, list):
            items = self.items
        else:
            items = [self.items]
        t = None
        for item in items:
            if isinstance(item, Row):
                item.full_width = self.full_width / len(items)
        if self.align in ['LEFT', 'CENTER', 'RIGHT']:
            t = Table([items],hAlign=self.align, colWidths=[None], style=TableStyle([
                ('BACKGROUND', (0, 0), (-1, -1), colors.lightgrey),
                ('ALIGN', (0, 0), (-1, -1), self.align),
                
            ]))
        else:
            t = Table([items], colWidths=[letter[0]/len(self.items)]*len(self.items), style=TableStyle([
                ('BACKGROUND', (0, 0), (-1, -1), colors.lightgrey),
                
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ]+ [
                ('ALIGN', (ind, 0 ), (ind, 0), 'LEFT') if ind == 0 else ('ALIGN', (ind, 0 ), (ind, 0), 'CENTER') if ind < len(self.items) - 1 else ('ALIGN', (ind, 0 ), (ind, 0), 'RIGHT') for ind in range(len(self.items))
            ]))
        return t

resume.append(
    Row(
        items=[
            'Item 1',
            'Item 2',
            'Item 3',
            'Item 4'
        ],
        align='LEFT'
    ).render()
)
resume.append(HRFlowable(width=total_width, thickness=1, color=colors.black,spaceBefore=10))


# resume.append(t2)
doc.build(resume)

In [7]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

# 1. Setup Document
doc = SimpleDocTemplate("dynamic_flex.pdf", pagesize=letter)
PAGE_WIDTH, _ = letter
left_m, right_m = 50, 50
available_width = PAGE_WIDTH

# 2. The Inner Table with DYNAMIC widths
# colWidths=None means "shrink-wrap to content"
inner_data = [
    ['Short', 'Medium Column', 'The Longest Column Name'],
    ['A', 'B', 'C']
]
inner_table = Table(inner_data, colWidths=None) 

inner_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), colors.white),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
]))

# 3. The Outer Table (100% width container)
outer_data = [[inner_table]]
outer_container = Table(outer_data, colWidths=[available_width])

outer_container.setStyle(TableStyle([
    # The '100% width' background
    ('BACKGROUND', (0, 0), (-1, -1), colors.black), 
    
    # This centers the dynamic inner table inside the full-width strip
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    
    # Internal spacing (Padding)
    ('TOPPADDING', (0, 0), (-1, -1), 15),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 15),
    ('LEFTPADDING', (0, 0), (-1, -1), 0),
    ('RIGHTPADDING', (0, 0), (-1, -1), 0),
]))

# Build the PDF
story = [outer_container]
doc.build(story)

In [5]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

# 1. Setup Document
doc = SimpleDocTemplate("full_width_container.pdf", pagesize=letter, 
                        leftMargin=50, rightMargin=50)
available_width = letter[0] # Page width minus margins
styles = getSampleStyleSheet()
story = []

# 2. Create the INNER Table (The content)
inner_data = [
    ['Item', 'Qty', 'Price'],
    ['Widget A', '2', '$10.00'],
    ['Widget B', '5', '$25.00']
]
inner_table = Table(inner_data, colWidths=[100, 50, 80])
inner_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), colors.white), # Cells are white
    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
]))

# 3. Create the OUTER Table (The Full-Width Container)
# Note: inner_table is wrapped in [[]] because Table data must be a list of lists
outer_data = [[inner_table]]
outer_container = Table(outer_data, colWidths=[available_width])

# 4. Style the Container
outer_container.setStyle(TableStyle([
    # The "Div" background color
    ('BACKGROUND', (0, 0), (-1, -1), colors.lavender), 
    
    # Center the inner table horizontally and vertically within the container
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    
    # Padding creates the "gap" between the container edge and the table
    ('TOPPADDING', (0, 0), (-1, -1), 0),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 0),
    
    # Remove default padding to ensure it reaches the margins perfectly
    ('LEFTPADDING', (0, 0), (-1, -1), 0),
    ('RIGHTPADDING', (0, 0), (-1, -1), 0),
]))

story.append(outer_container)
doc.build(story)

In [9]:
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from typing import Union, Literal
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from dataclasses import dataclass
doc = SimpleDocTemplate("test.pdf", pagesize=letter, topMargin=0, leftMargin=0, rightMargin=0, bottomMargin=0)
innerTable = Table([['Cell 1', 'Cell 2', 'Cell 3']])

@dataclass
class Direction:
    left: int
    right: int
    top: int
    bottom: int

class Row:
    def __init__(self, items: Union[list, str], valign: Literal['TOP', 'MIDDLE', 'BOTTOM'] = 'TOP', full_width: bool = False, width: float = None, height:float = None, align: Literal['LEFT', 'CENTER', 'RIGHT', 'BETWEEN', 'AROUND'] = 'LEFT', padding: Direction = Direction(0,0,0,0), margin: Direction = Direction(0,0,0,0), backgroundColor: str = '#ffffff'):
        self.items = items
        self.valign = valign
        self.full_width = full_width
        self.width = width
        self.height = height
        self.align = align
        self.padding = padding
        self.margin = margin
        self.backgroundColor = backgroundColor
        pass
    
    def render(self):
        print(f"Width Row: {self.width}")
        if self.width is not None:
            items = self.items
            for item in items:
                if hasattr(item, 'width'):
                    item.width = self.width / len(self.items) if hasattr(item, 'width') else None
                # print(hasattr(item , 'render'))
            self.items = [item.render() if hasattr(item, 'render') else item for item in items]
        align = [('ALIGN', (ind, 0 ), (ind, 0), 'LEFT') if ind == 0 else ('ALIGN', (ind, 0 ), (ind, 0), 'CENTER') if ind < len(self.items) - 1 else ('ALIGN', (ind, 0 ), (ind, 0), 'RIGHT') for ind in range(len(self.items))] if self.align == 'BETWEEN' else [('ALIGN', (ind, 0 ), (ind, 0), 'CENTER')  for ind in range(len(self.items))] if self.align == 'AROUND' else []
        
        halign = None
        colWidth = self.width/len(self.items) if self.width is not None else None
        if self.align in ['LEFT', 'CENTER', 'RIGHT']:
            halign = self.align
            colWidth = [None]*len(self.items)
        return Table([[item for item in self.items]] if isinstance(self.items, list) else [[self.items]], rowHeights=[self.height], colWidths=colWidth, hAlign=halign, style=TableStyle([
            ('LEFTPADDING', (0, 0), (-1, -1), self.padding.left),
            ('RIGHTPADDING', (0, 0), (-1, -1), self.padding.right),
            ('TOPPADDING', (0, 0), (-1, -1), self.padding.top),
            ('BOTTOMPADDING', (0, 0), (-1, -1), self.padding .bottom),
            ('VALIGN', (0, 0), (-1, -1), self.valign),
            ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(self.backgroundColor)),
        ]+align))

class Column:
    def __init__(self, items: Union[list, str], full_height: bool = False, width: float = None, height:float = None, align: Literal['LEFT', 'CENTER', 'RIGHT', 'BETWEEN', 'AROUND'] = 'LEFT', padding: Direction = Direction(0,0,0,0), margin: Direction = Direction(0,0,0,0), backgroundColor: str = '#ffffff'):
        self.items = items
        self.full_height = full_height
        self.width = width
        self.height = height
        self.align = align
        self.padding = padding
        self.margin = margin
        self.backgroundColor = backgroundColor
        pass
    
    def render(self):
        if self.width is not None:
            items = self.items
            for item in items:
                if hasattr(item, 'width'):
                    item.width = self.width / len(self.items)
                # print(hasattr(item , 'render'))
            self.items = [[item.render()] if hasattr(item, 'render') else [item] for item in items]
        return Table([item for item in self.items] if isinstance(self.items, list) else [[self.items]], 
                     rowHeights=[self.height]*len(self.items), colWidths=[self.width], 
                     style=TableStyle([
                        ('LEFTPADDING', (0, 0), (-1, -1), self.padding.left),
                        ('RIGHTPADDING', (0, 0), (-1, -1), self.padding.right),
                        ('TOPPADDING', (0, 0), (-1, -1), self.padding.top),
                        ('BOTTOMPADDING', (0, 0), (-1, -1), self.padding .bottom),
                        ('ALIGN', (0, 0), (-1, -1), self.align),
                        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(self.backgroundColor)),
                ]))

        

class FlexDiv:
    def __init__(self, items, backgroundColor: str = '#ffffff', width: float = None, height: float = None, orientation: Literal['ROW', 'COLUMN'] = 'ROW', padding: Direction = Direction(0,0,0,0), margin: Direction = Direction(0,0,0,0), valign: Literal['TOP', 'MIDDLE', 'BOTTOM'] = 'TOP', align: Literal['LEFT', 'CENTER', 'RIGHT', 'BETWEEN', 'AROUND']='LEFT'):
        self.items = items
        self.width = width
        self.height = height
        self.orientation = orientation
        self.backgroundColor = backgroundColor
        self.padding = padding
        self.valign = valign
        self.margin = margin
        self.align = align

    def render(self):
        if self.width is not None:
            items = self.items
            for item in items:
                if hasattr(item, 'width'):
                    item.width = self.width / len(self.items)
                # print(hasattr(item , 'render'))
            self.items = [item.render() if hasattr(item, 'render') else item for item in items]
        return Table([self.items],rowHeights=[self.height], colWidths=[self.width], style=TableStyle([
            ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(self.backgroundColor)),
            ('LEFTPADDING', (0, 0), (-1, -1), self.padding.left),
            ('RIGHTPADDING', (0, 0), (-1, -1), self.padding.right),
            ('TOPPADDING', (0, 0), (-1, -1), self.padding.top),
            ('BOTTOMPADDING', (0, 0), (-1, -1), self.padding .bottom),
            ('ALIGN', (0, 0), (-1, -1), self.align),
            ('VALIGN', (0, 0), (-1, -1), self.valign),
        ]))

resume = [
    FlexDiv(
        items=[
            Row(items=[
                Column(items=[
                        # Row(items=['Item 1', 'Item 2'], align='RIGHT')
                        FlexDiv(items=['Item 6', 'Item 7'], align='LEFT'),
                    ],
                    backgroundColor="#dfdfdf",
                    align='LEFT'), 
                 'Item 3', 'Item 4', 'Item 5'], 
                backgroundColor="#dfdfdf", 
                padding=Direction(10,10,10,10), 
                align='BETWEEN')],
        backgroundColor="#ffffff", 
        width=letter[0], 
        height=780, 
        padding=Direction(0,0,0,0), 
        align='LEFT', 
        valign='TOP')
    .render()
]

doc.build(resume)

Width Row: 612.0


In [2]:
%pip install reportlab

  Using cached reportlab-4.4.7-py3-none-any.whl.metadata (1.7 kB)
Using cached reportlab-4.4.7-py3-none-any.whl (2.0 MB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -------- ------------------------------- 1.6/7.0 MB 9.4 MB/s eta 0:00:01
   ----------------------- ---------------- 4.2/7.0 MB 10.5 MB/s eta 0:00:01
   ----------------------------------- ---- 6.3/7.0 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 9.9 MB/s  0:00:00

   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [pillow]
   ---------------------------------------- 0/3 [p

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.platypus.paragraph import ParagraphStyle
